In [2]:
from datetime import datetime
import pandas_datareader.data as web
import json
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import os

In [12]:
data_atual = datetime.now().strftime('%Y-%m-%d')
df_csv = pd.read_csv('dados/cotacoes.csv', sep=',')
data_do_arquivo = max(df_csv['Date'])
if data_do_arquivo == data_atual:
    print("Arquivo já está atualizado para a data atual.")
    #sys.exit(0) # Interrompe o script com sucesso
else:
    print("nao atualizado",data_do_arquivo)

nao atualizado 2024-05-06


In [9]:
df_csv.head(5)

,Ticker,Date,Open,High,Low,Close,Adj Close,Volume
0,RAIL3.SA,2015-04-02,13.800000,17.200001,13.8,17.200001,17.043989,2668180.0
1,RAIL3.SA,2015-04-06,17.600000,18.500000,17.1,17.400000,17.242172,1519710.0
2,RAIL3.SA,2015-04-07,16.799999,16.900000,15.6,15.800000,15.656686,2354260.0
3,RAIL3.SA,2015-04-08,16.100000,16.200001,15.5,15.600000,15.458500,1826330.0
4,RAIL3.SA,2015-04-09,15.400000,16.900000,15.3,16.299999,16.152149,2230390.0


In [23]:
def escrever_data_no_arquivo(caminho, data):

                with open(caminho, 'w') as f:
                    json.dump({'data': data}, f)
                print("Atualizando o arquivo para a data atual.")
#escrever_data_no_arquivo(caminho_arquivo, data_atual)
escrever_data_no_arquivo(caminho_arquivo, data_atual)

Atualizando o arquivo para a data atual.


In [2]:
# Lê arquivo csv com dados das minhas compras (não são dados históricos)
#df_tickers = pd.read_csv('codigo/quant/dados/tickers.csv', sep=';')
df_tickers = pd.read_csv('dados/tickers.csv', sep=';')
# Remover a linha onde o campo ticker é igual a "ENBR3.SA"
df_tickers = df_tickers[df_tickers['ticker'] != 'ENBR3.SA']
df_tickers.head(10)

,ticker,quantidade,paguei,investi,data
0,RAIL3.SA,57,"1,55",1590,1/1/2021
1,B3SA3.SA,1000,"11,80",11800,14/5/2015
2,CSUD3.SA,3000,"8,60",25800,14/5/2015
3,CMIG4.SA,1418,"28,50",40413,14/5/2015
5,PETR4.SA,250,"37,00",9250,14/5/2015
6,SHUL4.SA,1400,"16,50",23100,14/5/2015
7,VALE3.SA,100,"46,00",4600,26/1/2019


In [3]:
import os
# Lê todas cotacoes históricas gravadas no csv
#df_csv = pd.read_csv('codigo/quant/dados/cotacoes_historicas.csv', sep=',')
df_csv = pd.read_csv('dados/cotacoes_historicas.csv', sep=',')
#compara proveniente do csv com o dataframe que acabamos de baixar de cotações históricas



In [4]:
    datainicio = '2015-01-01'
    datafim = datetime.now()
    #datafim = '2023-11-14'
    # Lista de TICKERS
    #tickers = ['PETR4.SA','VALE3.SA','B3SA3.SA','RAIL3.SA','CARD3.SA','SHUL4.SA','ITUB4.SA','JBSS3.SA']
    tickers = df_tickers['ticker'].tolist()

    # Dataframe para preencher
    df_resumo = pd.DataFrame(columns=['ticker', 'datain', 'datafim', 'acao', 'valor_maximo_ticker','data_valor_maximo', 'valor_minimo_ticker', 'data_valor_minimo', 'valor_atual_ticker', 'valor_atual_total'])

    #função para obter valor atual
    def valor_atual(ticker):
        try:
            acao = yf.Ticker(ticker)
            valor_atual_ticker = acao.info["currentPrice"]
            return valor_atual_ticker
        except KeyError:
            return "Informação de preço atual não disponível para o ticker fornecido."
        except Exception as e:
            return f"Ocorreu um erro: {e}"


    for ticker in tickers:
        datafim = datetime.now()
        datainicio = df_tickers.loc[df_tickers['ticker'] == ticker, 'data'].values[0]
        datainicio = datetime.strptime(datainicio, '%d/%m/%Y')
        #@#
        # print(ticker,datainicio, datafim)
        acao = yf.download(ticker, start = datainicio, end = datafim)
        valor_maximo_ticker = acao['High'].max()
        if not acao[acao['High'] == valor_maximo_ticker].empty:
            data_valor_maximo = acao[acao['High'] == valor_maximo_ticker].index[0].date()
        else:
            data_valor_maximo = "null"  # ou um valor padrão, caso não haja valores
            # Filtrar valores acima de zero
        valores_acima_de_zero = acao['Low'][acao['Low'] > 0]
        if not valores_acima_de_zero.empty:
                valor_minimo_ticker = valores_acima_de_zero.min()
        else:
                valor_minimo_ticker = None  # ou um valor padrão, caso não haja valores acima de zero

        #data_valor_minimo = acao['Low'].idxmin().date()
        if not acao[acao['Low'] == valor_minimo_ticker].empty:
            data_valor_minimo = acao[acao['Low'] == valor_minimo_ticker].index[0].date()
        else:
            data_valor_minimo = "null"  # ou um valor padrão, caso não haja valores
            # Filtrar valores acima de zero
        acao = yf.Ticker(ticker)
        #  valor_atual_ticker = acao.info["currentPrice"]


        # current value of the ticker
        valor_atual_ticker = valor_atual(ticker)


        valor_atual_total = df_tickers.loc[df_tickers['ticker'] == ticker, 'quantidade'].values[0] * valor_atual(ticker)

        nova_linha = pd.DataFrame([[ticker, datainicio, datafim, acao, valor_maximo_ticker, data_valor_maximo, valor_minimo_ticker,data_valor_minimo, valor_atual_ticker, valor_atual_total]], columns=df_resumo.columns)

        df_resumo = pd.concat([df_resumo, nova_linha], ignore_index=True)



[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


In [6]:

#cotacao atual de um ticker B3
'''
import yfinance as yf
ticker = "B3SA3.SA"
acao = yf.Ticker(ticker)
valor_atual_ticker = acao.info["currentPrice"]
print(f"Cotação atual de B3SA3: {valor_atual_ticker}")
'''



'\nimport yfinance as yf\nticker = "B3SA3.SA"\nacao = yf.Ticker(ticker)\nvalor_atual_ticker = acao.info["currentPrice"]\nprint(f"Cotação atual de B3SA3: {valor_atual_ticker}")\n'

In [7]:
    #Cotação de um ticker para uma data informada:
    '''
    import yfinance as yf
    from datetime import datetime, timedelta

    ticker = "B3SA3.SA"
    #data_ontem = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
    data_desejada = "2023-08-14"
    acao = yf.Ticker(ticker)
    historico = acao.history(period="1d", interval="1d", start=data_desejada)
    valor_cotacao = historico["Close"].values[0] if not historico.empty else None

    if valor_cotacao is not None:
        print(f"Cotação de {ticker} em ({data_desejada}): {valor_cotacao:.2f}")
    else:
        print(f"Não há dados de cotação disponíveis para {ticker} em ({data_desejada}).")
        '''



'\nimport yfinance as yf\nfrom datetime import datetime, timedelta\n\nticker = "B3SA3.SA"\n#data_ontem = (datetime.today() - timedelta(days=1)).strftime(\'%Y-%m-%d\')\ndata_desejada = "2023-08-14"\nacao = yf.Ticker(ticker)\nhistorico = acao.history(period="1d", interval="1d", start=data_desejada)\nvalor_cotacao = historico["Close"].values[0] if not historico.empty else None\n\nif valor_cotacao is not None:\n    print(f"Cotação de {ticker} em ({data_desejada}): {valor_cotacao:.2f}")\nelse:\n    print(f"Não há dados de cotação disponíveis para {ticker} em ({data_desejada}).")\n    '

In [8]:
# Junta com uma analise e escreve no analise.csv
df_analise = df_resumo.merge(df_tickers, on='ticker', how='inner')
df_analise['lucro_prej'] = df_analise['valor_atual_total'] - df_analise['investi']
df_analise['lucro_prej_max'] = (df_analise['valor_maximo_ticker'] * df_analise['quantidade']) - df_analise['investi']
df_analise['lucro_prej_min'] = (df_analise['valor_minimo_ticker'] * df_analise['quantidade']) - df_analise['investi']
# update the field valor_atual_ticker with the value of the ticker now

path_csv = "dados/analise.csv"

df_analise.to_csv(path_csv)

#df_analise
df_analise[['ticker', 'datain', 'datafim', 'valor_maximo_ticker', 'data_valor_maximo', 'valor_minimo_ticker', 'data_valor_minimo', 'valor_atual_ticker', 'valor_atual_total', 'quantidade', 'paguei', 'investi', 'data', 'lucro_prej']]


#obter valor máximo
def obter_valor_maximo(ticker, periodo):
    # Obtém os dados históricos do ticker
    dados = yf.download(ticker, period=periodo)

# Obtém o valor máximo e a data correspondente
    valor_maximo = dados['Close'].max()
    data_valor_maximo = dados[dados['Close'] == valor_maximo].index[0].strftime('%Y-%m-%d')

    return valor_maximo, data_valor_maximo

#@#
#print(f'O valor máximo do ticker {ticker} foi de {valor_max} em {data_max}.')

nomes_colunas =  df_tickers.columns

# Imprimir os nomes das colunas
#@#
#print(nomes_colunas)







In [9]:
# Obter a data atual
data_atual = datetime.today().strftime("%Y%m%d")

# Definir o caminho antigo e novo
#caminho_antigo = 'dados/cotacoes_historicas.csv'
#caminho_novo = f'dados/cotacoes_historicas_{data_atual}.csv'

# Renomear o arquivo
#os.rename(caminho_antigo, caminho_novo)
#path_csv = "codigo/quant/dados/cotacoes_historicas.csv"
#path_csv = "dados/cotacoes_historicas.csv"

#df_cotacoes.to_csv(path_csv)

In [10]:
# Download de Dados históricos dos tickers
#!pip install yfinance --upgrade --no-cache-dir
tickers = df_tickers['ticker'].tolist()
# Crie uma lista vazia para armazenar os DataFrames de cotações
df_list = []
for ticker in tickers:
    datafim = datetime.now()
    datain = df_tickers.loc[df_tickers['ticker'] == ticker, 'data'].values[0]
    datain = datetime.strptime(datain, '%d/%m/%Y')
    acao = yf.download(ticker, start = datain, end = datafim)
    acao['Ticker'] = ticker  # Adicione uma coluna para identificar o ticker
    acao.reset_index(inplace=True)  # Resetar o índice, removendo a data como índice
    df_list.append(acao)

    # Concatene os DataFrames de cotações em um único DataFrame
    df_cotacoes = pd.concat(df_list)
    df_cotacoes.head(10)
    #compara_historico(df_cotacoes, df_tickers)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


In [11]:
#aqui embaixo no get estava dando erro TypeError: string indices must be integers
#é porque eu fiz import pandas_datareader as web ao inves de
#fazer o certo   import pandas_datareader.data as web
#Filtrando data na fonte
datainicio = '2015-01-01'
datafim = datetime.now()
#datafim = '2023-11-14'
# Lista de TICKERS
#tickers = ['PETR4.SA','VALE3.SA','B3SA3.SA','RAIL3.SA','CARD3.SA','SHUL4.SA','ITUB4.SA','JBSS3.SA']
tickers = df_tickers['ticker'].tolist()

In [12]:
# Dataframe para preencher
df_resumo = pd.DataFrame(columns=['ticker', 'datain', 'datafim', 'acao', 'valor_maximo_ticker','data_valor_maximo', 'valor_minimo_ticker', 'data_valor_minimo', 'valor_atual_ticker', 'valor_atual_total'])

#função para obter valor atual
def valor_atual(ticker):
    try:
        acao = yf.Ticker(ticker)
        valor_atual_ticker = acao.info["currentPrice"]
        return valor_atual_ticker
    except KeyError:
        return "Informação de preço atual não disponível para o ticker fornecido."
    except Exception as e:
        return f"Ocorreu um erro: {e}"

In [13]:


for ticker in tickers:
    datafim = datetime.now()
    datainicio = df_tickers.loc[df_tickers['ticker'] == ticker, 'data'].values[0]
    datainicio = datetime.strptime(datainicio, '%d/%m/%Y')
    #@#
    # print(ticker,datainicio, datafim)
    acao = yf.download(ticker, start = datainicio, end = datafim)
    valor_maximo_ticker = acao['High'].max()
    if not acao[acao['High'] == valor_maximo_ticker].empty:
        data_valor_maximo = acao[acao['High'] == valor_maximo_ticker].index[0].date()
    else:
        data_valor_maximo = "null"  # ou um valor padrão, caso não haja valores
        # Filtrar valores acima de zero
    valores_acima_de_zero = acao['Low'][acao['Low'] > 0]
    if not valores_acima_de_zero.empty:
            valor_minimo_ticker = valores_acima_de_zero.min()
    else:
            valor_minimo_ticker = None  # ou um valor padrão, caso não haja valores acima de zero

    #data_valor_minimo = acao['Low'].idxmin().date()
    if not acao[acao['Low'] == valor_minimo_ticker].empty:
        data_valor_minimo = acao[acao['Low'] == valor_minimo_ticker].index[0].date()
    else:
        data_valor_minimo = "null"  # ou um valor padrão, caso não haja valores
        # Filtrar valores acima de zero
    acao = yf.Ticker(ticker)
    #  valor_atual_ticker = acao.info["currentPrice"]


    # current value of the ticker
    valor_atual_ticker = valor_atual(ticker)


    valor_atual_total = df_tickers.loc[df_tickers['ticker'] == ticker, 'quantidade'].values[0] * valor_atual(ticker)

    nova_linha = pd.DataFrame([[ticker, datainicio, datafim, acao, valor_maximo_ticker, data_valor_maximo, valor_minimo_ticker,data_valor_minimo, valor_atual_ticker, valor_atual_total]], columns=df_resumo.columns)

    df_resumo = pd.concat([df_resumo, nova_linha], ignore_index=True)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


In [14]:
df_cotacoes.head(10)


,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
0,2021-01-04,19.360001,19.770000,19.059999,19.150000,18.976299,15041500,RAIL3.SA
1,2021-01-05,19.100000,19.379999,18.790001,19.299999,19.124939,10187600,RAIL3.SA
2,2021-01-06,19.110001,19.530001,18.910000,19.180000,19.006029,14303600,RAIL3.SA
3,2021-01-07,19.180000,19.799999,18.959999,19.629999,19.451944,16655700,RAIL3.SA
4,2021-01-08,19.730000,20.219999,19.680000,20.000000,19.818590,16484700,RAIL3.SA
5,2021-01-11,19.780001,20.030001,19.400000,19.650000,19.471764,11065800,RAIL3.SA
6,2021-01-12,19.990000,20.850000,19.680000,20.690001,20.502331,16763300,RAIL3.SA
7,2021-01-13,20.830000,20.830000,20.190001,20.700001,20.512241,7102400,RAIL3.SA
8,2021-01-14,20.760000,21.040001,20.580000,20.740000,20.551876,7145300,RAIL3.SA
9,2021-01-15,20.500000,21.379999,20.459999,21.209999,21.017614,15699200,RAIL3.SA


In [15]:
# Verificar se os DataFrames são iguais
if len(df_csv) == len(df_cotacoes) and pd.to_datetime(df_csv["Date"].max()).date() == pd.to_datetime(df_cotacoes["Date"].max()).date() and set(df_csv['Ticker']) == set(df_cotacoes['Ticker']) :
    print("DATAFRAMES IGUAIS, O número de linhas e a data de atualização é igual nos dois DataFrames e ambos tem os mesmos tickers.")
    print("Data do último registro: "+df_csv["Date"].max())
    print("Data hoje: " + datafim.strftime('%Y-%m-%d'))
# Se o dataframe atual é maior que o csv e tem data máxima mais recente, então renomeia o csv e substitui o csv pelo dataframe atual
elif len(df_csv) < len(df_cotacoes) and pd.to_datetime(df_csv["Date"].max()).date() < pd.to_datetime(df_cotacoes["Date"].max()).date():
    print("DATAFRAMES DIFERENTES.")
    print(df_csv["Date"].max())
    print(df_cotacoes["Date"].max().date())
    print(len(df_csv))
    print(len(df_cotacoes))



DATAFRAMES IGUAIS, O número de linhas e a data de atualização é igual nos dois DataFrames e ambos tem os mesmos tickers.
Data do último registro: 2024-07-31
Data hoje: 2024-07-31


In [16]:
def compara_historico(df_cotacoes, df_tickers):
    import os
    try:
        # Lê todas cotacoes históricas gravadas no csv
        df_csv = pd.read_csv('dados/cotacoes_historicas.csv', sep=',')
        # Verificar se os DataFrames são iguais
        if not df_csv.empty and not df_cotacoes.empty and df_csv.equals(df_cotacoes):
            #@#
            print("DATAFRAMES IGUAIS, O número de linhas e a data de atualização é igual nos dois DataFrames e ambos tem os mesmos tickers.")
            print("Data do último registro: " + str(df_csv["Date"].max()))
            print("Data hoje: " + str(datetime.now().date()))
        else:
            # Se o dataframe atual é maior que o csv e tem data máxima mais recente, então renomeia o csv e substitui o csv pelo dataframe atual
            if not df_csv.empty and not df_cotacoes.empty and len(df_csv) < len(df_cotacoes) and pd.to_datetime(df_csv["Date"].max()).date() < pd.to_datetime(df_cotacoes["Date"].max()).date():
                # Obter a data atual
                data_atual = datetime.today().strftime("%Y%m%d")
                # Definir o caminho antigo e novo
                caminho_antigo = 'dados/cotacoes_historicas.csv'
                caminho_novo = f'dados/cotacoes_historicas_{data_atual}.csv'
                # Renomear o arquivo
                os.rename(caminho_antigo, caminho_novo)
                # Substituir o arquivo pelo dataframe atual
                df_cotacoes.to_csv(caminho_antigo, index=False)
                # Faz uma análise dos tickers, joga tudo em df_resumo
                import requests
    except FileNotFoundError:
        print("Arquivo de cotacoes_historicas.csv não encontrado.")
    except Exception as e:
        print(f"Erro inesperado: {str(e)}")

In [17]:
compara_historico(df_cotacoes, df_tickers)

In [18]:
df_cotacoes.head(10)

,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
0,2021-01-04,19.360001,19.770000,19.059999,19.150000,18.976299,15041500,RAIL3.SA
1,2021-01-05,19.100000,19.379999,18.790001,19.299999,19.124939,10187600,RAIL3.SA
2,2021-01-06,19.110001,19.530001,18.910000,19.180000,19.006029,14303600,RAIL3.SA
3,2021-01-07,19.180000,19.799999,18.959999,19.629999,19.451944,16655700,RAIL3.SA
4,2021-01-08,19.730000,20.219999,19.680000,20.000000,19.818590,16484700,RAIL3.SA
5,2021-01-11,19.780001,20.030001,19.400000,19.650000,19.471764,11065800,RAIL3.SA
6,2021-01-12,19.990000,20.850000,19.680000,20.690001,20.502331,16763300,RAIL3.SA
7,2021-01-13,20.830000,20.830000,20.190001,20.700001,20.512241,7102400,RAIL3.SA
8,2021-01-14,20.760000,21.040001,20.580000,20.740000,20.551876,7145300,RAIL3.SA
9,2021-01-15,20.500000,21.379999,20.459999,21.209999,21.017614,15699200,RAIL3.SA


In [19]:
# Certifique-se de que a coluna 'Date' esteja no formato datetime
df_cotacoes['Date'] = pd.to_datetime(df_cotacoes['Date'])

# Ordene o DataFrame pela coluna 'Date' em ordem decrescente
df_cotacoes_sorted = df_cotacoes.sort_values(by='Date', ascending=False)

# Imprima o DataFrame ordenado
df_cotacoes_sorted.head(40)



,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
1370,2024-07-31,60.910000,61.240002,60.759998,61.070000,61.070000,8492900,VALE3.SA
2293,2024-07-31,19.350000,19.549999,19.350000,19.500000,19.500000,1800,CSUD3.SA
891,2024-07-31,22.059999,22.309999,22.049999,22.129999,22.129999,679700,RAIL3.SA
2293,2024-07-31,6.200000,6.260000,6.180000,6.240000,6.240000,26800,SHUL4.SA
2293,2024-07-31,10.850000,10.940000,10.710000,10.910000,10.910000,1462100,CMIG4.SA
2293,2024-07-31,37.049999,37.330002,37.040001,37.139999,37.139999,6265900,PETR4.SA
2293,2024-07-31,10.780000,10.860000,10.710000,10.720000,10.720000,6024700,B3SA3.SA
2292,2024-07-30,10.780000,10.840000,10.680000,10.760000,10.760000,21917500,B3SA3.SA
1369,2024-07-30,61.029999,61.110001,60.200001,60.220001,60.220001,28562300,VALE3.SA
890,2024-07-30,22.260000,22.350000,21.980000,22.190001,22.190001,5311200,RAIL3.SA


In [20]:
df_cotacoes_sorted.to_csv('dados/cotacoes_historicas.csv', index=False)

In [21]:
df_csv = pd.read_csv('dados/cotacoes_historicas.csv', sep=',')
#verifica arquivo original lido do primeiro csv
df_csv_sorted = df_csv.sort_values(by='Date', ascending=False)

# Imprima o DataFrame ordenado
df_csv_sorted.head(40)

,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
0,2024-07-31,60.910000,61.240002,60.759998,61.070000,61.070000,8492900,VALE3.SA
4,2024-07-31,10.850000,10.940000,10.710000,10.910000,10.910000,1462100,CMIG4.SA
6,2024-07-31,10.780000,10.860000,10.710000,10.720000,10.720000,6024700,B3SA3.SA
5,2024-07-31,37.049999,37.330002,37.040001,37.139999,37.139999,6265900,PETR4.SA
1,2024-07-31,19.350000,19.549999,19.350000,19.500000,19.500000,1800,CSUD3.SA
3,2024-07-31,6.200000,6.260000,6.180000,6.240000,6.240000,26800,SHUL4.SA
2,2024-07-31,22.059999,22.309999,22.049999,22.129999,22.129999,679700,RAIL3.SA
7,2024-07-30,10.780000,10.840000,10.680000,10.760000,10.760000,21917500,B3SA3.SA
11,2024-07-30,36.509998,36.770000,36.389999,36.650002,36.650002,16040700,PETR4.SA
13,2024-07-30,19.400000,19.400000,19.040001,19.350000,19.350000,27900,CSUD3.SA
